# Dynamic micropatterning: cell spreading, polarity, area and circularity

Part of the dataset for: Dar, Tesoro-Moreno, Palaia et al. *Caging of membrane-to-cortex attachment proteins can trigger cellular symmetry breaking.*

The notebook starts from the cleaned masks in `2_Segmentations/` (one stack per cell, 31 frames, 445 × 445 pixels, one frame every 2 min) and makes every table and plot of the analysis:

| Section | Output (in `4_Analysis_Code/output/`) |
|---|---|
| 1. Radial profiles and tables | `All_DataFrame_SINGLETIMEPOINTS.csv`, `All_DataFrame.csv`, `All_DataFrame_SPREADAREA.csv` |
| 2. Single-cell plots, frames 0–14 (0–30 min) | `Single_Cell_Plots/` |
| 3. Example cells | `Example_Cells/` |
| 4. Average heat maps per condition | `Average_Heat_Maps/` |
| 5. Polarity index | `Polarity_Index/` |
| 6. Radial plots per condition | `Polarity_Index/` |
| 7. Animation frames (optional) | `Polarity_Index/Animation_Frames/` |
| 8. Area and circularity | `Area_Circularity/` |
| 9. How the masks were made (optional) | needs the probability maps, which are not deposited |

Open the notebook from `4_Analysis_Code/` and run all cells. Nothing in `1_Raw/`, `2_Segmentations/` or `3_Processed/` is overwritten.

In [ ]:
import glob
import os
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tifffile
from matplotlib import cm, colors as mcolors
from scipy import ndimage
from scipy.stats import f_oneway, levene, mannwhitneyu, shapiro, ttest_ind
from skimage import measure
from statsmodels.stats.multicomp import pairwise_tukeyhsd

## Settings

In [ ]:
MASK_FOLDER = "../2_Segmentations"
OUTPUT_FOLDER = "output"

PIXEL_SIZE_UM = 0.325
FRAME_INTERVAL_MIN = 2

N_ANGLES = 360             # one ray per degree
ALIGN_ANGLE = np.pi / 2    # the shortest ray of the reference frame is rotated to 90°
REFERENCE_FRAME = 20       # reference frame for the tables and heat maps (40 min)

SINGLE_CELL_FRAMES = 15    # frames 0-14, i.e. 0-30 min
SINGLE_CELL_REFERENCE_FRAME = 14
SINGLE_CELL_BINS = 24      # 15° bins

MAKE_SINGLE_CELL_PLOTS = True     # 4 plots per cell, about 1,700 files
MAKE_ANIMATION_FRAMES = False     # 3 x 31 tif frames for the movies
PMAP_FOLDER = None                # folder with the *_foreground_pmaps.tif and *_boundary_pmaps.tif files (section 9)

EXAMPLE_CELLS = ["250327_6FP_E11_NI_4-cell5", "250327_6FP_E11_Doxo_3-cell2"]

CONDITIONS = ["6FP_E11_NI", "6FP_E11_Dox", "11FP_B10_NI", "11FP_B10_Dox", "CAMoesin_Poly_NI", "CAMoesin_Poly_Dox"]

# Conditions, order and colours of the polarity plots
POLARITY_CONDITIONS = ["6FP_E11_NI", "CAMoesin_Poly_Dox", "6FP_E11_Dox", "11FP_B10_Dox"]
POLARITY_COLORS = {"6FP_E11_NI": "grey", "CAMoesin_Poly_Dox": "plum", "6FP_E11_Dox": "coral", "11FP_B10_Dox": "crimson"}

# Order and colours of the area and circularity plots
BOX_ORDER = ["CAMoesin_Poly_NI", "CAMoesin_Poly_Dox", "6FP_E11_NI", "6FP_E11_Dox", "11FP_B10_NI", "11FP_B10_Dox"]
BOX_COLORS = {"6FP_E11_NI": "#ffcd9f", "6FP_E11_Dox": "#f77a0d", "11FP_B10_NI": "#fba2a2", "11FP_B10_Dox": "#f23d14",
              "CAMoesin_Poly_NI": "#f7e2f9", "CAMoesin_Poly_Dox": "#ec6df5"}
LINE_COLORS = {"6FP_E11_NI": "#d9ff9f", "6FP_E11_Dox": "#ff9233", "11FP_B10_NI": "#a4ff96", "11FP_B10_Dox": "#f23d14",
               "CAMoesin_Poly_NI": "#d4f5f0", "CAMoesin_Poly_Dox": "#f5bcf9"}

folders = {name: os.path.join(OUTPUT_FOLDER, name) for name in
           ["Single_Cell_Plots", "Example_Cells", "Average_Heat_Maps", "Polarity_Index", "Area_Circularity"]}
for folder in folders.values():
    os.makedirs(folder, exist_ok=True)

mask_files = sorted(glob.glob(os.path.join(MASK_FOLDER, "*", "*.tif")), key=os.path.basename)
print(len(mask_files), "masks")


def read_mask(path):
    return (tifffile.imread(path) > 0).astype(np.uint8)


def condition_of(name):
    return next(c for c in CONDITIONS if c in name)


def cell_of(name):
    return re.sub(r"_cleaned_mask.*", "", os.path.basename(name))


def save(fig, path, close=False, **kwargs):
    fig.savefig(path, dpi=300, bbox_inches="tight", **kwargs)
    plt.close(fig) if close else plt.show()

## 1. Radial profiles and tables

For each cell the origin is the centroid of the mask in the first frame and stays fixed for the whole movie. For each of 360 angles, the radial distance is the furthest mask pixel along the ray, in pixels. A frame with an empty mask gives NaN.

To compare cells, the profiles of each cell are rotated together so that the shortest ray of a reference frame points to 90°. If the reference frame is empty, the first non-empty frame is used.

A cell with an empty first frame has no origin and is left out of the radial analysis (`250514_6FP_E11_NI_cell18`, so 419 of the 420 cells).

In [ ]:
def radial_profiles_over_time_fixed_centroid(mask_stack, n_angles=N_ANGLES):
    # Returns None if the first frame is empty
    T, H, W = mask_stack.shape
    if mask_stack[0].sum() == 0:
        return None
    angles = np.linspace(0, 2 * np.pi, n_angles, endpoint=False)
    y_c, x_c = ndimage.center_of_mass(mask_stack[0])
    radii = np.arange(1, max(H, W))
    x = np.round(x_c + np.outer(np.cos(angles), radii)).astype(int)
    y = np.round(y_c + np.outer(np.sin(angles), radii)).astype(int)
    inside = np.cumprod((x >= 0) & (x < W) & (y >= 0) & (y < H), axis=1).astype(bool)   # a ray stops at the image edge
    x, y = np.clip(x, 0, W - 1), np.clip(y, 0, H - 1)

    radial_distances = np.full((T, n_angles), np.nan)
    for t in range(T):
        if mask_stack[t].sum() == 0:
            continue
        hit = (mask_stack[t][y, x] == 1) & inside
        furthest = radii.size - 1 - np.argmax(hit[:, ::-1], axis=1)
        any_hit = hit.any(axis=1)
        radial_distances[t, any_hit] = radii[furthest[any_hit]]
    return angles, radial_distances


def align_radial_profiles(radial_data, angles, reference_frame=REFERENCE_FRAME, align_angle=ALIGN_ANGLE):
    if np.all(np.isnan(radial_data[reference_frame])):
        reference_frame = np.where(~np.all(np.isnan(radial_data), axis=1))[0][0]
    min_angle = angles[np.nanargmin(radial_data[reference_frame])]
    angle_shift = (align_angle - min_angle) % (2 * np.pi)
    shift_steps = int(round((angle_shift / (2 * np.pi)) * len(angles)))
    return np.roll(radial_data, shift_steps, axis=1), shift_steps


def angle_bins(n_bins):
    edges = np.linspace(0, 2 * np.pi, n_bins + 1)
    return edges, (edges[:-1] + edges[1:]) / 2


def max_spread_per_angle_bin(angles, radial_data, n_bins):
    # Maximum over time and over the angles of each bin
    edges, bin_centers = angle_bins(n_bins)
    max_spread = np.array([np.nanmax(radial_data[:, (angles >= edges[i]) & (angles < edges[i + 1])]) for i in range(n_bins)])
    return bin_centers, max_spread


def compute_spread_over_time(angles, radial_data, n_bins):
    # Maximum over the angles of each bin, per frame
    edges, bin_centers = angle_bins(n_bins)
    spread = np.full((radial_data.shape[0], n_bins), np.nan)
    for i in range(n_bins):
        in_bin = (angles >= edges[i]) & (angles < edges[i + 1])
        for t in range(radial_data.shape[0]):
            if np.any(~np.isnan(radial_data[t, in_bin])):
                spread[t, i] = np.nanmax(radial_data[t, in_bin])
    return bin_centers, spread


def spreading_area_over_time_fixed_centroid(mask_stack, n_bins=36):
    # Mask pixels per angular bin and frame, around the same fixed centroid
    T, H, W = mask_stack.shape
    if mask_stack[0].sum() == 0:
        return None
    edges, bin_centers = angle_bins(n_bins)
    y_c, x_c = ndimage.center_of_mass(mask_stack[0])
    Y, X = np.indices((H, W))
    pixel_bin = np.searchsorted(edges, (np.arctan2(Y - y_c, X - x_c) % (2 * np.pi)).ravel(), side="right") - 1
    area = np.full((T, n_bins), np.nan)
    for t in range(T):
        if mask_stack[t].sum() > 0:
            area[t] = np.bincount(pixel_bin, weights=mask_stack[t].ravel(), minlength=n_bins)[:n_bins]
    return bin_centers, area


def align_area_profiles(area_over_time, bin_centers, reference_frame=0, align_angle=ALIGN_ANGLE):
    # Rotates the bin with the smallest area of the reference frame to align_angle
    n_bins = len(bin_centers)
    angle_shift = (align_angle - bin_centers[np.nanargmin(area_over_time[reference_frame])]) % (2 * np.pi)
    return np.roll(area_over_time, int(round((angle_shift / (2 * np.pi)) * n_bins)), axis=1)


def area_and_circularity(mask):
    area = int(mask.sum())
    contours = measure.find_contours(mask, 0.5)
    perimeter = sum(np.sum(np.sqrt(np.sum(np.diff(c, axis=0) ** 2, axis=1))) for c in contours)
    return area, (4 * np.pi * area / perimeter ** 2 if perimeter > 0 else np.nan)

In [ ]:
distance_rows, max_spread_rows, spread_area_rows, area_rows = [], [], [], []
radial_by_cell = {}

for path in mask_files:
    filename, clone = os.path.basename(path), condition_of(os.path.basename(path))
    mask = read_mask(path)

    for t in range(mask.shape[0]):
        area, circularity = area_and_circularity(mask[t])
        area_rows.append((filename, cell_of(path)[:6], clone, t, area, area * PIXEL_SIZE_UM ** 2, circularity))

    result = radial_profiles_over_time_fixed_centroid(mask)
    if result is None:
        print("No radial profile, first frame is empty:", filename)
        continue
    angles, radial_data = result
    radial_by_cell[filename] = radial_data
    aligned, _ = align_radial_profiles(radial_data, angles)
    T = aligned.shape[0]

    distance_rows.append(pd.DataFrame({"filename": filename, "clone_name": clone, "time": np.repeat(np.arange(T), N_ANGLES),
                                       "angle": np.tile(angles, T), "distance": aligned.ravel()}))

    bin_centers, max_spread = max_spread_per_angle_bin(angles, aligned, N_ANGLES)
    max_spread_rows.append(pd.DataFrame({"filename": filename, "clone_name": clone, "max_spread": max_spread, "angle": bin_centers}))

    bin_centers, area_over_time = spreading_area_over_time_fixed_centroid(mask)
    aligned_area = align_area_profiles(area_over_time, bin_centers)
    spread_area_rows.append(pd.DataFrame({"filename": filename, "clone_name": clone, "time": np.repeat(np.arange(T), len(bin_centers)),
                                          "angle": np.tile(bin_centers, T), "area": aligned_area.ravel()}))

spread_data_all = pd.concat(distance_rows, ignore_index=True)
spread_data_max = pd.concat(max_spread_rows, ignore_index=True)
spread_area_all = pd.concat(spread_area_rows, ignore_index=True)
area_data = pd.DataFrame(area_rows, columns=["filename", "date", "condition", "timepoint", "area", "area_um2", "circularity"])

for table, name in [(spread_data_all, "All_DataFrame_SINGLETIMEPOINTS.csv"), (spread_data_max, "All_DataFrame.csv"), (spread_area_all, "All_DataFrame_SPREADAREA.csv")]:
    table["angle_deg"] = np.degrees(table["angle"])
    table.to_csv(os.path.join(OUTPUT_FOLDER, name), index=False)
area_data.to_csv(os.path.join(folders["Area_Circularity"], "Cell_Area_Circularity_per_frame.csv"), index=False)

print(spread_data_all["filename"].nunique(), "cells with radial profiles,", area_data["filename"].nunique(), "cells with area and circularity")
print(spread_data_all.groupby("clone_name")["filename"].nunique())

## 2. Single-cell plots, frames 0–14 (0–30 min)

Four plots per cell, aligned on frame 14: the radial profiles over time, the maximum spread per 15° bin, and the maximum spread per bin over time, with and without centring on 90°.

In [ ]:
def plot_radial_evolution(angles, radial_data, save_path, cmap_name="plasma"):
    T = radial_data.shape[0]
    cmap = plt.get_cmap(cmap_name)
    fig = plt.figure(figsize=(7, 7))
    ax = plt.subplot(111, polar=True)
    for t, color in enumerate(cmap(np.linspace(0, 1, T))):
        ax.plot(angles, radial_data[t], color=color, lw=1)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=T - 1))
    sm.set_array([])
    fig.colorbar(sm, ax=ax, pad=0.1).set_label("Timepoint")
    ax.set_title("Radial Profiles Over Time (Aligned)", va="bottom")
    save(fig, save_path, close=True)


def plot_radial_bar(bin_centers, max_spread, save_path, title="Maximum Cell Spread per Angle Bin"):
    fig = plt.figure(figsize=(7, 7))
    ax = fig.add_subplot(111, polar=True)
    ax.bar(bin_centers, max_spread, width=2 * np.pi / len(bin_centers), bottom=0.0, alpha=0.7)
    ax.set_title(title, va="bottom")
    save(fig, save_path, close=True)


def plot_spread_heatmap(spread_over_time, bin_centers, save_path, title="Spread Over Time per Angle Bin"):
    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(spread_over_time.T, aspect="auto", cmap="viridis", origin="lower", extent=[0, spread_over_time.shape[0], 0, 2 * np.pi])
    fig.colorbar(im, ax=ax, label="Radial Distance")
    if len(bin_centers) <= 36:
        ax.set_yticks(bin_centers)
        ax.set_yticklabels([f"{np.degrees(a):.0f}°" for a in bin_centers])
    ax.set_xlabel("Time Frame")
    ax.set_ylabel("Angle")
    ax.set_title(title)
    fig.tight_layout()
    save(fig, save_path, close=True)


def plot_spread_heatmap_aligned(spread_over_time, bin_centers, save_path, title="Cell Spread Over Time (90° Centered)"):
    bin_centers_deg = np.degrees(bin_centers)
    shift = spread_over_time.shape[1] // 2 - np.argmin(np.abs(bin_centers_deg - 90))
    spread_shifted = np.roll(spread_over_time, shift, axis=1)
    centers_shifted = np.roll(bin_centers_deg, shift)
    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(spread_shifted.T, aspect="auto", cmap="viridis", origin="lower",
                   extent=[0, spread_over_time.shape[0], centers_shifted[0], centers_shifted[-1]])
    ax.set_ylim(centers_shifted[0], centers_shifted[-1])
    ax.axhline(90, color="white", linestyle="--", linewidth=1)
    yticks = np.arange(0, 361, 45)
    yticks = yticks[(yticks >= centers_shifted[0]) & (yticks <= centers_shifted[-1])]
    ax.set_yticks(yticks)
    ax.set_yticklabels([f"{int(a)}°" for a in yticks])
    fig.colorbar(im, ax=ax, label="Radial Distance")
    ax.set_xlabel("Time Frame")
    ax.set_ylabel("Angle (°)")
    ax.set_title(title)
    fig.tight_layout()
    save(fig, save_path, close=True)


if MAKE_SINGLE_CELL_PLOTS:
    for filename, radial_data in radial_by_cell.items():
        base = os.path.splitext(filename)[0]
        out = lambda suffix: os.path.join(folders["Single_Cell_Plots"], base + suffix)
        aligned, _ = align_radial_profiles(radial_data[:SINGLE_CELL_FRAMES], angles, reference_frame=SINGLE_CELL_REFERENCE_FRAME)
        plot_radial_evolution(angles, aligned, out("_aligned_radial_profile.png"))
        bin_centers, max_spread = max_spread_per_angle_bin(angles, aligned, SINGLE_CELL_BINS)
        plot_radial_bar(bin_centers, max_spread, out("_max_spread_polar_bar_t0_30.png"))
        bin_centers, spread_over_time = compute_spread_over_time(angles, aligned, SINGLE_CELL_BINS)
        plot_spread_heatmap(spread_over_time, bin_centers, out("_max_spread_heatmap_t0_30.png"))
        plot_spread_heatmap_aligned(spread_over_time, bin_centers, out("_max_spread_heatmap_ALIGNED_t0_30.png"))

## 3. Example cells

The same plots over all 31 frames, with 1° bins for the heat maps, plus the outline of the mask in every frame coloured by time.

In [ ]:
def plot_area_heatmap(area_over_time, save_path, title="Aligned Spreading Area per Angle Bin"):
    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(area_over_time.T, aspect="auto", cmap="magma", origin="lower", extent=[0, area_over_time.shape[0], 0, 360])
    fig.colorbar(im, ax=ax, label="Area (pixels)")
    ax.set_xlabel("Time Frame")
    ax.set_ylabel("Angle (°)")
    ax.set_title(title)
    fig.tight_layout()
    save(fig, save_path, close=True)


def plot_mask_outlines(mask_stack, save_path):
    T, H, W = mask_stack.shape
    total_width = W + int(W * 0.1)
    fig = plt.figure(figsize=(total_width / 100, H / 100), dpi=100)
    ax = fig.add_axes([0, 0, W / total_width, 1])
    ax.set_xlim(0, W)
    ax.set_ylim(H, 0)
    ax.axis("off")
    for t, color in enumerate(plt.cm.jet(np.linspace(0, 1, T))):
        for contour in measure.find_contours(mask_stack[t], 0.5):
            ax.plot(contour[:, 1], contour[:, 0], color=color, linewidth=1)
    sm = plt.cm.ScalarMappable(cmap=plt.cm.jet, norm=plt.Normalize(vmin=0, vmax=T - 1))
    sm.set_array([])
    fig.colorbar(sm, cax=fig.add_axes([W / total_width + 0.01, 0.05, 0.02, 0.9])).set_label("Timepoint")
    fig.savefig(save_path, dpi=100, bbox_inches="tight", pad_inches=0)
    plt.close(fig)


for cell in EXAMPLE_CELLS:
    path = next(p for p in mask_files if cell_of(p) == cell)
    base = os.path.splitext(os.path.basename(path))[0]
    out = lambda suffix: os.path.join(folders["Example_Cells"], base + suffix)
    mask = read_mask(path)
    radial_data = radial_by_cell[os.path.basename(path)]
    aligned, _ = align_radial_profiles(radial_data, angles)

    fig = plt.figure(figsize=(6, 6))
    plt.polar(angles, aligned[30])
    plt.title("Aligned Profile at t=30")
    save(fig, out("_polar_profile_t30.svg"), close=True)
    plot_radial_evolution(angles, aligned, out("_radial_evolution.svg"))
    bin_centers, max_spread = max_spread_per_angle_bin(angles, aligned, SINGLE_CELL_BINS)
    plot_radial_bar(bin_centers, max_spread, out("_max_spread_bar.svg"), title="Maximum Cell Spread per Angular Bin")
    bin_centers, spread_over_time = compute_spread_over_time(angles, aligned, N_ANGLES)
    plot_spread_heatmap(spread_over_time, bin_centers, out("_spread_heatmap.svg"), title="Spread Over Time")
    plot_spread_heatmap_aligned(spread_over_time, bin_centers, out("_spread_heatmap_aligned.svg"), title="Spread Over Time (90° Centered)")
    bin_centers, area_over_time = spreading_area_over_time_fixed_centroid(mask, n_bins=N_ANGLES)
    plot_area_heatmap(align_area_profiles(area_over_time, bin_centers), out("_area_heatmap.svg"))
    plot_mask_outlines(mask, out("_cleaned_mask_with_bar.svg"))

## 4. Average heat maps per condition

- `heatmap_<condition>`: mean radial distance per 10° bin and frame.
- `heatmap_mean_<condition>` and `heatmap_std_<condition>`: mean and standard deviation, with one colour scale per measure across the conditions.
- `max_spread_per_condition`: mean maximum spread per angle and condition, for all six conditions and for the subset of four.

In [ ]:
def clone_heatmap_tables(n_bins=36):
    data = spread_data_all[["clone_name", "time", "angle", "distance"]].copy()
    # Angles are shifted by +90° and wrapped into [0, 2π)
    data["angle_bin"] = pd.cut((data["angle"] + np.pi / 2) % (2 * np.pi), bins=np.linspace(0, 2 * np.pi, n_bins + 1), labels=False, include_lowest=True)
    grouped = data.groupby(["clone_name", "time", "angle_bin"])["distance"]
    return grouped.mean().unstack().reindex(columns=range(n_bins)), grouped.std().unstack().reindex(columns=range(n_bins))


def plot_clone_heatmap(table, clone, cmap, label, title, name, vmin, vmax):
    grouped = table.loc[clone]
    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(grouped.T, aspect="auto", cmap=cmap, origin="lower", extent=[grouped.index.min(), grouped.index.max(), 0, 360], vmin=vmin, vmax=vmax)
    fig.colorbar(im, ax=ax, label=label)
    ax.set_xlabel("Time Frame")
    ax.set_ylabel("Angle (°)")
    ax.set_title(f"{title} — {clone} (Shifted +90°)")
    for ext in ("png", "svg"):
        fig.savefig(os.path.join(folders["Average_Heat_Maps"], f"{name}_{clone}.{ext}"), dpi=300, bbox_inches="tight")
    plt.show()


heat_mean, heat_std = clone_heatmap_tables()
for clone in CONDITIONS:
    plot_clone_heatmap(heat_mean, clone, "viridis", "Average Radial Distance", "Average Spread Heatmap", "heatmap", heat_mean.min().min(), heat_mean.max().max())
    plot_clone_heatmap(heat_mean, clone, "viridis", "Average Radial Distance", "Mean Spread Heatmap", "heatmap_mean", heat_mean.min().min(), heat_mean.max().max())
    plot_clone_heatmap(heat_std, clone, "magma", "Std Dev of Radial Distance", "Std Dev Spread Heatmap", "heatmap_std", heat_std.min().min(), heat_std.max().max())

In [ ]:
def plot_max_spread_per_condition(order, name):
    pivot = (spread_data_max.groupby(["clone_name", "angle_deg"])["max_spread"].mean().reset_index()
             .pivot(index="clone_name", columns="angle_deg", values="max_spread"))
    columns = pivot.columns.to_numpy()
    pivot = pivot.reindex(columns=np.roll(columns, int(90 / (columns[1] - columns[0])))).reindex(order)   # shifted by +90°

    fig = plt.figure(figsize=(10, 5))
    sns.heatmap(pivot, cmap="viridis")
    shifted = pivot.columns.to_numpy()
    labels = np.arange(90, 450, 20) % 360
    plt.xticks([np.argmin(np.abs(shifted - a)) for a in labels], [f"{int(a)}°" for a in labels], rotation=45)
    plt.xlabel("Angle (deg, shifted +90°)")
    plt.ylabel("Clone")
    plt.title("Max Spread Heatmap per Clone (Shifted +90°)")
    plt.tight_layout()
    save(fig, os.path.join(folders["Average_Heat_Maps"], name))


plot_max_spread_per_condition(BOX_ORDER, "max_spread_per_condition.svg")
plot_max_spread_per_condition(POLARITY_CONDITIONS, "max_spread_per_condition_SUBSET.svg")

## 5. Polarity index

For each cell and frame, the radial distances are divided by their sum over the 360 angles, and

polarity index = (max − min) / (max + min)

of these normalised distances. The comparison at 10 min uses a one-way ANOVA with Tukey's post hoc test, and a two-sided Mann–Whitney U test for 6FP E11 NI vs 11FP B10 Dox.

In [ ]:
polarity_data = spread_data_all[spread_data_all["clone_name"].isin(POLARITY_CONDITIONS)].copy()
polarity_data["time_min"] = polarity_data["time"] * FRAME_INTERVAL_MIN
polarity_data["distance_norm"] = polarity_data["distance"] / polarity_data.groupby(["filename", "time_min"])["distance"].transform("sum")

pi_df = (polarity_data.groupby(["filename", "clone_name", "time_min"])
         .agg(max_dist=("distance_norm", "max"), min_dist=("distance_norm", "min")).reset_index())
pi_df["polarization_index"] = (pi_df["max_dist"] - pi_df["min_dist"]) / (pi_df["max_dist"] + pi_df["min_dist"])
pi_df.to_csv(os.path.join(folders["Polarity_Index"], "Polarity_Index_per_cell.csv"), index=False)

pi_stats = (pi_df.groupby(["clone_name", "time_min"])["polarization_index"]
            .agg(mean="mean", sem=lambda x: x.std() / np.sqrt(len(x))).reset_index())
pi_stats.to_csv(os.path.join(folders["Polarity_Index"], "Polarity_Index_mean_sem.csv"), index=False)
timepoints_min = sorted(pi_df["time_min"].unique())

In [ ]:
def plot_polarity_over_time(data, errorbar, name, figsize, xlim=30, markers=False):
    fig = plt.figure(figsize=figsize)
    sns.lineplot(data=data, x="time_min", y="polarization_index", hue="clone_name", palette=POLARITY_COLORS, linewidth=2,
                 errorbar=errorbar, marker="o" if markers else None, seed=0)
    plt.xlabel("Time (min)", fontsize=12)
    plt.ylabel("Polarization Index", fontsize=12)
    plt.ylim(0, 1)
    plt.xlim(0, xlim)
    if markers:
        plt.title("Polarization Index Over Time per Clone", fontsize=14)
        plt.xticks(timepoints_min)
        plt.grid(True, linestyle="--", alpha=0.5)
    else:
        plt.title("Polarization Index Over Time per Clone t 0 to t 30", fontsize=14)
    plt.legend(title="Clone", fontsize=10)
    plt.tight_layout()
    save(fig, os.path.join(folders["Polarity_Index"], name))


subset = pi_df[pi_df["time_min"] <= 30]
plot_polarity_over_time(subset, "se", "Polarity_Index_Over_Time_SEM.svg", (5, 4))
plot_polarity_over_time(subset, "sd", "Polarity_Index_Over_Time_SDEV.svg", (5, 4))
for xlim in (30, 60):
    plot_polarity_over_time(pi_df, ("ci", 68), f"Polarity_Index_Over_Time_SEM_0_{xlim}.svg", (6, 4), xlim=xlim, markers=True)   # 68% CI, close to the SEM
    plot_polarity_over_time(pi_df, "sd", f"Polarity_Index_Over_Time_SDEV_0_{xlim}.svg", (6, 4), xlim=xlim, markers=True)

# Polarity index at 10 min
pi_df_t = pi_df[pi_df["time_min"] == 10].dropna(subset=["polarization_index"]).reset_index(drop=True)
fig = plt.figure(figsize=(5, 4))
sns.violinplot(data=pi_df_t, x="clone_name", y="polarization_index", order=POLARITY_CONDITIONS, hue="clone_name", palette=POLARITY_COLORS, legend=False, cut=0, inner=None, linewidth=1)
sns.boxplot(data=pi_df_t, x="clone_name", y="polarization_index", order=POLARITY_CONDITIONS, width=0.2, showcaps=True, boxprops={"facecolor": "none", "zorder": 3},
            showfliers=False, whiskerprops={"linewidth": 1}, medianprops={"color": "black", "linewidth": 2})
sns.swarmplot(data=pi_df_t, x="clone_name", y="polarization_index", order=POLARITY_CONDITIONS, hue="clone_name", palette=POLARITY_COLORS, legend=False, size=4, alpha=0.6, edgecolor="black", linewidth=0.3)
plt.ylabel("Polarity index")
plt.xlabel("")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
save(fig, os.path.join(folders["Polarity_Index"], "Polarity_Index_violin_t10.svg"))

print(pi_df_t.groupby("clone_name")["polarization_index"].count())
F_stat, p_value = f_oneway(*[pi_df_t.loc[pi_df_t["clone_name"] == c, "polarization_index"] for c in POLARITY_CONDITIONS])
print("ANOVA F =", F_stat, "p =", p_value)
tukey = pairwise_tukeyhsd(endog=pi_df_t["polarization_index"], groups=pi_df_t["clone_name"], alpha=0.05)
print(tukey)
U, p_mw = mannwhitneyu(pi_df_t.loc[pi_df_t["clone_name"] == "6FP_E11_NI", "polarization_index"],
                       pi_df_t.loc[pi_df_t["clone_name"] == "11FP_B10_Dox", "polarization_index"], alternative="two-sided")
print("Mann-Whitney 6FP_E11_NI vs 11FP_B10_Dox: U =", U, "p =", p_mw)

with open(os.path.join(folders["Polarity_Index"], "Polarity_Index_statistics_t10.txt"), "w") as f:
    f.write(f"Cells per condition at 10 min:\n{pi_df_t.groupby('clone_name')['polarization_index'].count().to_string()}\n\n")
    f.write(f"One-way ANOVA: F = {F_stat}, p = {p_value}\n\nTukey HSD:\n{tukey}\n\n")
    f.write(f"Mann-Whitney U, two-sided, 6FP_E11_NI vs 11FP_B10_Dox: U = {U}, p = {p_mw}\n")

## 6. Radial plots per condition

Mean normalised distance per angle, per condition and time.

In [ ]:
def mean_profile(sub):
    avg = sub.groupby("angle_deg", as_index=False).agg(mean_dist=("distance_norm", "mean")).sort_values("angle_deg")
    return np.deg2rad(avg["angle_deg"].values), avg["mean_dist"].values


def closed_profile(sub):
    angles_, values = mean_profile(sub)
    return np.append(angles_, angles_[0]), np.append(values, values[0])


def format_polar(ax):
    ax.set_theta_zero_location("E")
    ax.set_theta_direction(1)
    ax.set_yticklabels([])


# Conditions side by side, selected times
timepoints, line_colors = [0, 10, 20, 30], ["grey", "#d7b5d8", "#df65b0", "#ce1256"]
fig, axes = plt.subplots(1, 4, subplot_kw={"projection": "polar"}, figsize=(20, 5))
handles, labels = [], []
for ax, clone in zip(axes, POLARITY_CONDITIONS):
    for t, color in zip(timepoints, line_colors):
        angles_, values = mean_profile(polarity_data[(polarity_data["clone_name"] == clone) & (polarity_data["time_min"] == t)])
        line, = ax.plot(angles_, values, linewidth=2, color=color)
        ax.fill(angles_, values, color=color, alpha=0.2)
        if clone == POLARITY_CONDITIONS[0]:
            handles.append(line)
            labels.append(f"t={t}")
    ax.set_title(clone)
    format_polar(ax)
fig.legend(handles, labels, title="Time (min)", loc="lower center", ncol=len(timepoints), bbox_to_anchor=(0.5, 1.05))
save(fig, os.path.join(folders["Polarity_Index"], "Radial_Plots_Comparisons.svg"))

# All times up to 60 min, coloured by time
timepoints = [t for t in timepoints_min if t <= 60]
norm = mcolors.Normalize(vmin=min(timepoints), vmax=max(timepoints))
fig, axes = plt.subplots(1, 4, subplot_kw={"projection": "polar"}, figsize=(20, 5))
for ax, clone in zip(axes, POLARITY_CONDITIONS):
    for t in timepoints:
        angles_, values = mean_profile(polarity_data[(polarity_data["clone_name"] == clone) & (polarity_data["time_min"] == t)])
        ax.plot(angles_, values, linewidth=2, color=cm.rainbow(norm(t)))
        ax.fill(angles_, values, color=cm.rainbow(norm(t)), alpha=0.25)
    ax.set_title(clone)
    format_polar(ax)
sm = cm.ScalarMappable(norm=norm, cmap=cm.rainbow)
sm.set_array([])
fig.colorbar(sm, ax=axes, orientation="vertical", fraction=0.05, pad=0.1).set_label("Time (min)")
save(fig, os.path.join(folders["Polarity_Index"], "Radial_Plots_Time_Rainbow.svg"))

# Radial bars, 30° bins
timepoints, bar_colors, bin_size = [0, 10, 20], ["lightgrey", "#41b6c4", "#225ea8"], 30
degree_bins = np.arange(0, 360 + bin_size, bin_size)
fig, axes = plt.subplots(1, 4, figsize=(20, 5), subplot_kw={"projection": "polar"}, constrained_layout=True)
handles, labels = [], []
for ax, clone in zip(axes, POLARITY_CONDITIONS):
    for t, color in zip(timepoints, bar_colors):
        sub = polarity_data[(polarity_data["clone_name"] == clone) & (polarity_data["time_min"] == t)].copy()
        sub["angle_bin"] = pd.cut(sub["angle_deg"], bins=degree_bins, labels=degree_bins[:-1], include_lowest=True, right=False)
        binned = sub.groupby("angle_bin", as_index=False, observed=False)["distance_norm"].mean()
        bar = ax.bar(np.deg2rad(binned["angle_bin"].astype(float) + bin_size / 2), binned["distance_norm"].values, width=np.deg2rad(bin_size * 0.99), color=color, alpha=0.5, edgecolor="k")
        if clone == POLARITY_CONDITIONS[0]:
            handles.append(bar[0])
            labels.append(f"t={t}")
    ax.set_title(clone, fontsize=12)
    format_polar(ax)
    ax.set_ylim(0, 0.004)
fig.legend(handles, labels, title="Time (min)", loc="lower center", ncol=len(timepoints), bbox_to_anchor=(0.5, 1.05))
save(fig, os.path.join(folders["Polarity_Index"], "Radial_Plots_Comparisons_BINNED.svg"))

# Times side by side, all conditions on one plot
timepoints = [0, 10, 20, 30, 40]
fig, axes = plt.subplots(1, 5, figsize=(25, 5), subplot_kw={"projection": "polar"}, constrained_layout=True)
handles, labels = [], []
for ax, t in zip(axes, timepoints):
    for clone in POLARITY_CONDITIONS:
        angles_, values = closed_profile(polarity_data[(polarity_data["clone_name"] == clone) & (polarity_data["time_min"] == t)])
        line, = ax.plot(angles_, values, linewidth=2, color=POLARITY_COLORS[clone])
        ax.fill(angles_, values, color=POLARITY_COLORS[clone], alpha=0.5)
        if ax is axes[0]:
            handles.append(line)
            labels.append(clone)
    ax.set_title(f"t = {t} min", fontsize=12, pad=12)
    format_polar(ax)
fig.legend(handles, labels, title="Clone", loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.05))
save(fig, os.path.join(folders["Polarity_Index"], "Radial_Plots_Comparisons_CLONES_TIME.svg"))

## 7. Animation frames (optional)

Set `MAKE_ANIMATION_FRAMES = True` to save one tif per time point for the three movies: the polarity index over time, the normalised radial profile per time point, and the cumulative radial profile. The frames are numbered, so they can be opened as an image sequence in Fiji and saved as a movie.

In [ ]:
if MAKE_ANIMATION_FRAMES:
    frames_folder = os.path.join(folders["Polarity_Index"], "Animation_Frames")
    for name in ("Polarity_Index", "Radial_Norm", "Radial_Norm_Cumulative"):
        os.makedirs(os.path.join(frames_folder, name), exist_ok=True)
    norm = mcolors.Normalize(vmin=min(timepoints_min), vmax=max(timepoints_min))

    for frame_idx, t_now in enumerate(timepoints_min):
        # Polarity index up to this time
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.set_xlim(min(timepoints_min), max(timepoints_min))
        ax.set_ylim(0, 1)
        ax.set_xlabel("Time (min)")
        ax.set_ylabel("Polarization Index")
        ax.grid(True, linestyle="--", alpha=0.5)
        for clone in POLARITY_CONDITIONS:
            sub = pi_stats[(pi_stats["clone_name"] == clone) & (pi_stats["time_min"] <= t_now)].sort_values("time_min")
            ax.plot(sub["time_min"], sub["mean"], color=POLARITY_COLORS[clone], lw=2, marker="o", label=clone)
            ax.fill_between(sub["time_min"], sub["mean"] - sub["sem"], sub["mean"] + sub["sem"], color=POLARITY_COLORS[clone], alpha=0.25)
        ax.set_title(f"Cumulative Polarization Index (t ≤ {t_now} min)")
        ax.legend(title="Clone", fontsize=9, loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
        save(fig, os.path.join(frames_folder, "Polarity_Index", f"PI_cumulative_{frame_idx:03d}_t{t_now}.tif"), close=True)

        # Radial profile at this time, and all times up to this one
        for cumulative, name, title, alpha in ((False, "Radial_Norm", f"Time = {t_now} min", 0.3),
                                               (True, "Radial_Norm_Cumulative", f"Cumulative polarity up to t = {t_now} min", 0.15)):
            fig, axes = plt.subplots(1, 4, subplot_kw={"projection": "polar"}, figsize=(20, 5), constrained_layout=True)
            for ax, clone in zip(axes, POLARITY_CONDITIONS):
                for t in (timepoints_min[:frame_idx + 1] if cumulative else [t_now]):
                    angles_, values = closed_profile(polarity_data[(polarity_data["clone_name"] == clone) & (polarity_data["time_min"] == t)])
                    ax.plot(angles_, values, color=cm.viridis(norm(t)), lw=2)
                    ax.fill(angles_, values, color=cm.viridis(norm(t)), alpha=alpha)
                ax.set_title(clone)
                format_polar(ax)
                ax.set_ylim(0, 0.004)
            fig.suptitle(title, fontsize=16)
            fig.savefig(os.path.join(frames_folder, name, f"frame_{frame_idx:03d}_t{t_now}.tif"), dpi=300)
            plt.close(fig)

## 8. Area and circularity

Area is the number of mask pixels × (0.325 µm)². Circularity is 4π·area / perimeter², with the perimeter from the contours of the mask. Frames are numbered from 0, one frame every 2 min. A frame with an empty mask has an area of 0 and no circularity.

In [ ]:
def plot_over_time(y, ylabel, title, name, figsize=(6, 4), ylim=None, xlim=None):
    fig = plt.figure(figsize=figsize)
    sns.lineplot(data=area_data, x="timepoint", y=y, hue="condition", palette=LINE_COLORS, estimator="mean", errorbar="se", linewidth=2)
    if ylim:
        plt.ylim(*ylim)
    if xlim:
        plt.xlim(*xlim)
    plt.xlabel("Timepoint")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend(title="Condition")
    plt.tight_layout()
    for ext in name[1]:
        fig.savefig(os.path.join(folders["Area_Circularity"], f"{name[0]}.{ext}"), dpi=300, bbox_inches="tight")
    plt.show()


def plot_box_timepoints(y, timepoints, ylabel, title, name, ylim=None, reference_line=True, figsize=(9, 5), width=0.6):
    data = area_data[area_data["timepoint"].isin(timepoints)]
    fig = plt.figure(figsize=figsize)
    sns.boxplot(data=data, x="timepoint", y=y, hue="condition", palette=BOX_COLORS, hue_order=BOX_ORDER, width=width, fliersize=0)
    n = data["condition"].nunique()
    if reference_line:
        reference = data[(data["condition"] == "CAMoesin_Poly_NI") & (data["timepoint"] == timepoints[0])][y].mean()
        plt.axhline(y=reference, color="grey", linestyle="--", linewidth=2, label=f"CAMoesin_Poly_NI mean = {reference:.2f}")
        n += 1
        plt.xlim(-0.35, len(timepoints) - 0.65)
    handles, labels = plt.gca().get_legend_handles_labels()
    plt.legend(handles[:n], labels[:n], title="Condition", bbox_to_anchor=(1.05, 1), loc="upper left")
    if ylim:
        plt.ylim(*ylim)
    plt.xlabel("Timepoint", fontsize=12, fontweight="bold")
    plt.ylabel(ylabel, fontsize=12, fontweight="bold")
    plt.title(title, fontsize=14, fontweight="bold")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    save(fig, os.path.join(folders["Area_Circularity"], name))


plot_over_time("area_um2", "Mask area (µm²)", "Cell area over time per condition", ("cell_area_over_time", ("svg", "png")), ylim=(800, 2200))
plot_over_time("circularity", "Circularity", "Cell circularity over time per condition", ("cell_circularity_over_time", ("svg", "png")), figsize=(5, 4), ylim=(0.2, 0.8))
plot_over_time("circularity", "Circularity", "Cell circularity over time per condition", ("cell_circularity_over_time_t0_15", ("svg",)), figsize=(5, 4), ylim=(0.2, 0.8), xlim=(0, 15))
plot_over_time("circularity", "Circularity", "Cell circularity over time per condition", ("cell_circularity_over_time_t0_30", ("svg",)), figsize=(5, 4), ylim=(0.2, 0.8), xlim=(0, 30))

plot_box_timepoints("circularity", [0, 7, 15], "Circularity", "Cell Circularity at Selected Timepoints", "cell_circularity_boxplot_timepoints.svg", ylim=(0, 1), reference_line=False, figsize=(10, 6), width=0.7)
plot_box_timepoints("circularity", [5, 20], "Circularity", "Cell Circularity at Selected Timepoints", "cell_circularity_two_time.svg", ylim=(0, 1))
plot_box_timepoints("area_um2", [5, 20], "Area (µm²)", "Cell Area at Selected Timepoints", "cell_area_boxplot_timepoints.svg")

### Statistics on the area

These repeat the tests of the original analysis, on the time points it used: ANOVA, Shapiro–Wilk, Levene and Tukey on frames 5 and 15, and Welch's t-test on frames 5 and 10. In both cases the two frames are pooled, so most cells count twice.

In [ ]:
def area_statistics(timepoints):
    data = area_data[area_data["timepoint"].isin(timepoints)]
    groups = {c: g["area_um2"] for c, g in data.groupby("condition")}
    F_stat, p_value = f_oneway(*groups.values())
    lines = [f"Frames {timepoints}", f"One-way ANOVA: F = {F_stat}, p = {p_value}"]
    lines += [f"Shapiro-Wilk {c}: p = {shapiro(g).pvalue}" for c, g in groups.items()]
    lines += [f"Levene: p = {levene(*groups.values()).pvalue}", "Tukey HSD:", str(pairwise_tukeyhsd(endog=data["area_um2"], groups=data["condition"], alpha=0.05))]
    return lines


data = area_data[area_data["timepoint"].isin([5, 10])]
welch = ttest_ind(data.loc[data["condition"] == "11FP_B10_NI", "area_um2"], data.loc[data["condition"] == "6FP_E11_NI", "area_um2"], equal_var=False)
lines = area_statistics([5, 15]) + ["", "Frames [5, 10]", f"Welch's t-test, 11FP_B10_NI vs 6FP_E11_NI: t = {welch.statistic}, p = {welch.pvalue}"]
print("\n".join(lines))
with open(os.path.join(folders["Area_Circularity"], "Cell_Area_statistics.txt"), "w") as f:
    f.write("\n".join(lines) + "\n")

## 9. How the masks were made (optional)

Masks were made from the phase-contrast channel (channel 0 of `1_Raw`) with a trained PlantSeg 3D U-Net, using time as the third dimension. The network gives a foreground and a boundary probability map per cell. The maximum of the two then goes through `preprocess_image_stack` below, frame by frame. Each cell's mask comes from one of three versions of the model and was selected by quality (see `Dynamic_micropatterning_segmentation_quality.xlsx`).

The probability maps are not part of this dataset, so this cell only runs if `PMAP_FOLDER` is set. It saves the masks to `output/Cleaned_Masks/`.

In [ ]:
from scipy.ndimage import binary_fill_holes
from skimage.filters import threshold_otsu
from skimage.morphology import closing, disk, remove_small_objects
from skimage.segmentation import clear_border


def preprocess_image_stack(img_prediction, min_size=1000, closing_radius=7):
    # Otsu threshold, largest object, small objects and objects touching the border removed, holes filled, closing
    cleaned_stack = np.zeros_like(img_prediction, dtype=bool)
    for t in range(img_prediction.shape[0]):
        img = img_prediction[t]
        labeled = measure.label(img > threshold_otsu(img))
        props = measure.regionprops(labeled)
        largest = labeled == max(props, key=lambda p: p.area).label if len(props) > 1 else labeled > 0
        cleaned = clear_border(remove_small_objects(largest, min_size=min_size))
        cleaned_stack[t] = binary_fill_holes(closing(binary_fill_holes(cleaned), disk(closing_radius)))
    return cleaned_stack


if PMAP_FOLDER:
    out_folder = os.path.join(OUTPUT_FOLDER, "Cleaned_Masks")
    os.makedirs(out_folder, exist_ok=True)
    for path in sorted(glob.glob(os.path.join(PMAP_FOLDER, "*_foreground_pmaps.tif"))):
        cell = os.path.basename(path).replace("_foreground_pmaps.tif", "")
        pmaps = np.maximum(tifffile.imread(path), tifffile.imread(path.replace("_foreground_", "_boundary_")))
        tifffile.imwrite(os.path.join(out_folder, f"{cell}_cleaned_mask.tif"), preprocess_image_stack(pmaps).astype(np.uint8) * 255)